In [11]:
import json
import pandas as pd
import re
from tqdm import tqdm
from boto3

# ---- CONFIGURAÇÕES ---- #
ENDPOINT_NAME = "g5-llama32-ft-2025-07-28-18-15-12"
TEST_FILE = "test2.jsonl"
OUTPUT_FILE = "predictions.jsonl"
CONTENT_TYPE = "application/json"

# ---- Inicializar cliente SageMaker ---- #
runtime = boto3.client("sagemaker-runtime")

def infer(prompt: str) -> str:
    """Faz inferência via SageMaker endpoint."""
    payload = json.dumps({
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 256,
            "temperature": 0.0,
            #"top_p": 1.0,
            "do_sample": False
        }
    })  # Formato correto para Llama (text-generation-inference)
    
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType=CONTENT_TYPE,
        Body=payload
    )
    result = response['Body'].read().decode("utf-8")
    return result


def clean_prompt(full_prompt: str) -> str:
    """Remove o bloco assistant e tudo o que vem depois."""
    split_token = "<|start_header_id|>assistant<|end_header_id|>"
    if split_token in full_prompt:
        return full_prompt.split(split_token)[0].strip()
    return full_prompt.strip()


def run_batch_inference(input_file: str, output_file: str):
    """Corre inferência no dataset todo e guarda os outputs."""
    results = []
    with open(input_file, "r") as f:
        lines = f.readlines()

    with open(output_file, "w") as out_f:
        for line in tqdm(lines, desc="Inferência em batch"):
            item = json.loads(line)
            raw_prompt = item["text"]
            task = item["task"]

            prompt = clean_prompt(raw_prompt)  # 🔧 limpa o assistant

            try:
                output = infer(prompt)
                result = {
                    "task": task,
                    "input_text": prompt,
                    "model_output": output
                }
                out_f.write(json.dumps(result) + "\n")
            except Exception as e:
                print(f"❌ Erro na inferência: {e}")
                continue
                
    print(json.dumps(result, indent=2))
    print(f"✅ Inferência concluída. Resultados guardados em: {output_file}")
    
if __name__ == "__main__":
    run_batch_inference(TEST_FILE, OUTPUT_FILE)

Inferência em batch: 100%|██████████| 1839/1839 [11:56<00:00,  2.57it/s]

{
  "task": "car_review_rating",
  "input_text": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 27 Jul 2025\n\nYou are an expert automotive assistant specialized in analyzing car reviews. Your tasks are to: 1. Classify the sentiment of the review as Positive, Neutral, or Negative. 2. Predict the numerical rating (1.000 to 5.000, where 1.000 is very negative and 5.000 is very positive). 3. If based on the review you consider it necessary, generate a professional, empathetic response to the review, addressing the reviewer's concerns or praise. 4. If the review indicates issues (e.g., Negative sentiment or low rating), provide an escalation plan to address the concerns, including specific actions for customer service or technical teams. Ensure your responses are concise, professional, and tailored to the review content.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nPlease analyze the following car review and provid

In [13]:
# Avaliação

import json
import pandas as pd
import re
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def extract_json_block(text):
    """Extrai bloco JSON de ```json {...}```"""
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return None
    return None

def extract_field_block(text, field):
    """Extrai valor de campos como '**Sentiment**: Positive'"""
    match = re.search(rf"\*\*{re.escape(field)}\*\*:\s*(.+)", text)
    if match:
        return match.group(1).strip()
    return None

# ---- Carregar ficheiros ---- #
with open("test2.jsonl") as f:
    gt_data = [json.loads(line) for line in f]

with open("predictions.jsonl") as f:
    pred_data = [json.loads(line) for line in f]

assert len(gt_data) == len(pred_data)

# ---- Avaliação ---- #
email_fields = ["Customer Name", "Car Model", "Pickup", "Dropoff"]

results = []

for gt, pred in zip(gt_data, pred_data):
    task = gt["task"]
    gt_text = gt["text"]
    pred_json = json.loads(pred["model_output"])
    pred_text = pred_json.get("generated_text", "")

    row_result = {"task": task}

    if task == "email_extraction":
        gt_json = extract_json_block(gt_text)
        pred_json = extract_json_block(pred_text)
        
        if not gt_json or not pred_json:
            row_result.update({f"{field}_match": False for field in email_fields})
            row_result.update({f"{field}_llm": "" for field in email_fields})
            row_result.update({f"{field}_regex": "" for field in email_fields})
            row_result["all_correct"] = False
        else:
            for field in email_fields:
                gt_val = str(gt_json.get(field, "")).strip()
                pred_val = str(pred_json.get(field, "")).strip()
                row_result[f"{field}_regex"] = gt_val
                row_result[f"{field}_llm"] = pred_val
                row_result[f"{field}_match"] = gt_val == pred_val
            row_result["all_correct"] = all(row_result[f"{field}_match"] for field in email_fields)

    elif task == "car_review_rating":
        # Extrair campos
        gt_sentiment = extract_field_block(gt_text, "Sentiment")
        gt_rating_str = extract_field_block(gt_text, "Rating")
        pred_sentiment = extract_field_block(pred_text, "Sentiment")
        pred_rating_str = extract_field_block(pred_text, "Rating")

        try:
            gt_rating = float(gt_rating_str)
            pred_rating = float(pred_rating_str)
            rating_error = abs(gt_rating - pred_rating)
        except (TypeError, ValueError):
            gt_rating = pred_rating = rating_error = None

        row_result.update({
            "gt_sentiment": gt_sentiment,
            "pred_sentiment": pred_sentiment,
            "sentiment_correct": gt_sentiment == pred_sentiment,
            "gt_rating": gt_rating,
            "pred_rating": pred_rating,
            "rating_error": rating_error
        })

    results.append(row_result)

# ---- Gerar DataFrame ---- #
df = pd.DataFrame(results)

# ---- Métricas para email_extraction ---- #
df_email = df[df["task"] == "email_extraction"]
diff_cols = []
for field in email_fields:
    diff_cols += [f"{field}_llm", f"{field}_regex", f"{field}_match"]

diff_cols = ["task"] + diff_cols
print(diff_cols)
print(df_email)
df_diff = df_email[~df_email["all_correct"]]
df_diff[diff_cols].to_csv("email_extraction_diferencas.csv", index=False, encoding="utf-8")


with open("relatorio_metricas_fine_tuning_emails.txt", "w", encoding="utf-8") as f:
    f.write("📬 MÉTRICAS - EMAIL EXTRACTION\n")
    f.write("="*50 + "\n")
    for field in email_fields:
        acc = df_email[f"{field}_match"].mean()
        f.write(f"- Accuracy em '{field}': {acc:.2%}\n")
    acc_total = df_email["all_match"].mean()
    f.write(f"- Accuracy total (todos os campos corretos): {acc_total:.2%}\n\n")


# ---- Métricas para car_review_rating ---- #
df_rating = df[df["task"] == "car_review_rating"]
acc_sentiment = df_rating["sentiment_correct"].mean()
mae_rating = mean_absolute_error(df_rating["gt_rating"], df_rating["pred_rating"])
rmse_rating = mean_squared_error(df_rating["gt_rating"], df_rating["pred_rating"])
r2 = round(r2_score(df_rating["gt_rating"], df_rating["pred_rating"]), 4)

with open("relatorio_metricas_fine_tuning_reviews.txt", "w", encoding="utf-8") as f:
    f.write("🚗 MÉTRICAS - CAR REVIEW RATING\n")
    f.write("="*50 + "\n")
    f.write(f"- Acurácia do sentimento: {acc_sentiment:.2%}\n")
    f.write(f"- MAE do rating: {mae_rating:.3f}\n")
    f.write(f"- RMSE do rating: {rmse_rating:.3f}\n")
    f.write(f"- Score R²: {r2}\n")


# Iniciar clientes S3 e SageMaker
#s3_client = boto3.client('s3', region_name='eu-west-1')
#sagemaker_runtime = boto3.client('sagemaker-runtime', region_name="eu-west-1")

#bucket_name = 'i32419'

#def upload_file(local_file_path, s3_path):
#    s3_client.upload_file(local_file_path, bucket_name, s3_path)
#    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")

# Enviar relatório para o S3
#upload_file("relatorio_metricas_fine_tuning_emails.txt", "output/relatorio_metricas_fine_tuning_emails.txt")
#upload_file("diferencas_detectadas.csv", "output/diferencas_detectadas.csv")
#upload_file("relatorio_metricas_fine_tuning_review.txt", "output/relatorio_metricas_fine_tuning_review.txt")

['task', 'Customer Name_llm', 'Customer Name_regex', 'Customer Name_match', 'Car Model_llm', 'Car Model_regex', 'Car Model_match', 'Pickup_llm', 'Pickup_regex', 'Pickup_match', 'Dropoff_llm', 'Dropoff_regex', 'Dropoff_match']
                  task gt_sentiment pred_sentiment sentiment_correct  \
22    email_extraction          NaN            NaN               NaN   
46    email_extraction          NaN            NaN               NaN   
53    email_extraction          NaN            NaN               NaN   
56    email_extraction          NaN            NaN               NaN   
63    email_extraction          NaN            NaN               NaN   
...                ...          ...            ...               ...   
1723  email_extraction          NaN            NaN               NaN   
1725  email_extraction          NaN            NaN               NaN   
1727  email_extraction          NaN            NaN               NaN   
1745  email_extraction          NaN            NaN    

KeyError: "None of [Index([-2, -1, -2, -2, -2, -2, -1, -2, -2, -2, -2, -2, -2, -2, -2, -1, -2, -2,\n       -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2,\n       -2, -2, -2, -1, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -1,\n       -1, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2,\n       -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -1, -2, -2, -2],\n      dtype='object')] are in the [columns]"

In [11]:
# Avaliação

import json
import pandas as pd
import re
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def extract_json_block(text):
    """Extrai bloco JSON de json {...}
"""
    match = re.search(r"json\s*(\{.*?\})\s*", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return None
    return None

def extract_field_block(text, field):
    """Extrai valor de campos como '**Sentiment**: Positive'"""
    match = re.search(rf"\*\*{re.escape(field)}\*\*:\s*(.+)", text)
    if match:
        return match.group(1).strip()
    return None

# ---- Carregar ficheiros ---- #
with open("test2.jsonl") as f:
    gt_data = [json.loads(line) for line in f]

with open("predictions.jsonl") as f:
    pred_data = [json.loads(line) for line in f]

assert len(gt_data) == len(pred_data)

# ---- Avaliação ---- #
email_fields = ["Customer Name", "Car Model", "Pickup", "Dropoff"]

results = []

for gt, pred in zip(gt_data, pred_data):
    task = gt["task"]
    gt_text = gt["text"]
    pred_json = json.loads(pred["model_output"])
    pred_text = pred_json.get("generated_text", "")

    row_result = {"task": task}

    if task == "email_extraction":
        gt_json = extract_json_block(gt_text)
        pred_json = extract_json_block(pred_text)
        
        if not gt_json or not pred_json:
            row_result.update({f"{field}_match": False for field in email_fields})
            row_result["all_match"] = False
        else:
            for field in email_fields:
                row_result[f"{field}_match"] = gt_json.get(field) == pred_json.get(field)
            row_result["all_match"] = all(row_result[f"{f}_match"] for f in email_fields)

    elif task == "car_review_rating":
        # Extrair campos
        gt_sentiment = extract_field_block(gt_text, "Sentiment")
        gt_rating_str = extract_field_block(gt_text, "Rating")
        pred_sentiment = extract_field_block(pred_text, "Sentiment")
        pred_rating_str = extract_field_block(pred_text, "Rating")

        try:
            gt_rating = float(gt_rating_str)
            pred_rating = float(pred_rating_str)
            rating_error = abs(gt_rating - pred_rating)
        except (TypeError, ValueError):
            gt_rating = pred_rating = rating_error = None

        row_result.update({
            "gt_sentiment": gt_sentiment,
            "pred_sentiment": pred_sentiment,
            "sentiment_correct": gt_sentiment == pred_sentiment,
            "gt_rating": gt_rating,
            "pred_rating": pred_rating,
            "rating_error": rating_error
        })

    results.append(row_result)

# ---- Gerar DataFrame ---- #
df = pd.DataFrame(results)

# ---- Métricas para email_extraction ---- #
df_email = df[df["task"] == "email_extraction"]
with open("relatorio_metricas_fine_tuning_emails.txt", "w", encoding="utf-8") as f:
    f.write("📬 MÉTRICAS - EMAIL EXTRACTION\n")
    f.write("="*50 + "\n")
    for field in email_fields:
        acc = df_email[f"{field}_match"].mean()
        f.write(f"- Accuracy em '{field}': {acc:.2%}\n")
    f.write(f" - Accuracy total (todos os campos corretos): {df_email['all_match'].mean():.2%}")

print("📬 Métricas para email_extraction:")
for field in email_fields:
    print(f" - Accuracy em '{field}': {acc:.2%}")
print(f" - Accuracy total (todos os campos corretos): {df_email['all_match'].mean():.2%}\n\n")


# ---- Métricas para car_review_rating ---- #
df_rating = df[df["task"] == "car_review_rating"]
acc_sentiment = df_rating["sentiment_correct"].mean()
mae_rating = mean_absolute_error(df_rating["gt_rating"], df_rating["pred_rating"])
mse_rating = mean_squared_error(df_rating["gt_rating"], df_rating["pred_rating"])
r2 = round(r2_score(df_rating["gt_rating"], df_rating["pred_rating"]), 4)
print("🚗 Métricas para car_review_rating:")
print(f" - Accuracy do sentimento: {acc_sentiment:.2%}")
print(f" - MAE do rating: {mae_rating:.3f}")
print(f" - MSE do rating: {rmse_rating:.3f}")
print(f" - Score R²: {r2}")

with open("relatorio_metricas_fine_tuning_reviews.txt", "w", encoding="utf-8") as f:
    f.write("🚗 MÉTRICAS - CAR REVIEW RATING\n")
    f.write("="*50 + "\n")
    f.write(f"- Accuracy do sentimento: {acc_sentiment:.2%}\n")
    f.write(f"- MAE do rating: {mae_rating:.3f}\n")
    f.write(f"- MSE do rating: {mse_rating:.3f}\n")
    f.write(f"- Score R²: {r2}\n")
    

# Iniciar clientes S3 e SageMaker
s3_client = boto3.client('s3', region_name='eu-west-1')
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name="eu-west-1")

bucket_name = 'i32419'

def upload_file(local_file_path, s3_path):
    s3_client.upload_file(local_file_path, bucket_name, s3_path)
    print(f"Arquivo {local_file_path} enviado para s3://{bucket_name}/{s3_path}")

# Enviar métricas para o S3
upload_file("relatorio_metricas_fine_tuning_emails.txt", "output/relatorio_metricas_fine_tuning_emails.txt")
upload_file("relatorio_metricas_fine_tuning_reviews.txt", "output/relatorio_metricas_fine_tuning_reviews.txt")

📬 Métricas para email_extraction:
 - Accuracy em 'Customer Name': 100.00%
 - Accuracy em 'Car Model': 100.00%
 - Accuracy em 'Pickup': 100.00%
 - Accuracy em 'Dropoff': 100.00%
 - Accuracy total (todos os campos corretos): 100.00%


🚗 Métricas para car_review_rating:
 - Accuracy do sentimento: 87.85%
 - MAE do rating: 0.432
 - MSE do rating: 0.496
 - Score R²: 0.4705
Arquivo relatorio_metricas_fine_tuning_emails.txt enviado para s3://i32419/output/relatorio_metricas_fine_tuning_emails.txt
Arquivo relatorio_metricas_fine_tuning_reviews.txt enviado para s3://i32419/output/relatorio_metricas_fine_tuning_reviews.txt
